In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV

In [ ]:
# Dictionary to store all model results
model_results = {}

In [3]:
# Not to use 'group. Instead use 'milkperiod','zdate','zdate_month'
df = pd.read_excel('E:\IUT\Lessons\Project-Bachelor\Husbandry\Dataset\TCI_sas (1).xlsx', sheet_name="Sheet1", usecols = ["milkperiod","zdate","zdate_month","firstmilk","firstmilkdays",
                                                                "prelendays","drylendays","milkdays",
                                                                "previous_Milk305",
                                                                "firstmilk_previous","SCS_305"])

In [4]:
df.isnull().sum()

milkperiod               0
zdate                    0
zdate_month              0
firstmilk                0
firstmilkdays            0
prelendays               0
drylendays            2059
milkdays              2059
previous_Milk305         0
firstmilk_previous       0
SCS_305                  0
dtype: int64

In [ ]:
df['drylendays'] = df['drylendays'].fillna(df['drylendays'].median())
df['milkdays'] = df['milkdays'].fillna(df['milkdays'].median())

In [6]:
df.isnull().sum()

milkperiod            0
zdate                 0
zdate_month           0
firstmilk             0
firstmilkdays         0
prelendays            0
drylendays            0
milkdays              0
previous_Milk305      0
firstmilk_previous    0
SCS_305               0
dtype: int64

### **Split the data into train, validation, and test sets**

In [7]:
# Separate features and target
X = df.drop(columns=["firstmilk"])
y = df["firstmilk"].astype(float) 

# First split: separate test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# Second split: separate train (70%) and validation (15%) from remaining data
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)  # 0.1765 ~ 15/(100-15)


### **Scale the features**

In [8]:
numerical_columns = ["milkperiod","zdate","zdate_month", "firstmilkdays", "prelendays", "drylendays", "milkdays",
                     "previous_Milk305", "firstmilk_previous", "SCS_305"]

scaler = StandardScaler()

# Fit scaler on training data only
scaler.fit(X_train[numerical_columns])

# Transform train, validation, and test sets
X_train[numerical_columns] = scaler.transform(X_train[numerical_columns])
X_val[numerical_columns] = scaler.transform(X_val[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

### **Define functions for additional metrics**


In [9]:
def calculate_mpe(y_true, y_pred):
    # Avoid division by zero
    mask = y_true != 0
    return np.mean((y_true[mask] - y_pred[mask]) / y_true[mask] * 100)

def calculate_smape(y_true, y_pred):
    # Avoid division by zero
    numerator = np.abs(y_true - y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator != 0
    return np.mean(numerator[mask] / denominator[mask] * 100)

def calculate_sdr(y_true, y_pred):
    return np.std(y_pred) / np.std(y_true)

# **Initial Random Forest model**

In [10]:
initial_model = RandomForestRegressor(random_state=42)
initial_model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

### **Prediction on test set**


In [11]:
y_test_pred = initial_model.predict(X_test)

### **Evaluate on test set**
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)

# Calculate additional metrics for initial model
test_mpe = calculate_mpe(y_test, y_test_pred)
test_smape = calculate_smape(y_test, y_test_pred)
test_sdr = calculate_sdr(y_test, y_test_pred)

print("=" * 25)
print("Test Set (RF_Initial):")
print(f"R²    : {test_r2:.4f}")
print(f"MAE   : {test_mae:.4f}")
print(f"RMSE  : {test_rmse:.4f}")
print(f"MPE   : {test_mpe:.4f}")
print(f"sMAPE : {test_smape:.4f}")
print(f"SDR   : {test_sdr:.4f}")
print("=" * 25)

Test Set (RF_Initial):
R²    : 0.3477
MAE   : 6.8285
RMSE  : 8.9583
MPE   : -6.9363
sMAPE : 17.2230
SDR   : 0.6106


### **Prediction on all datasets**


In [ ]:
y_train_pred = initial_model.predict(X_train)
y_val_pred = initial_model.predict(X_val)
y_test_pred = initial_model.predict(X_test)

### **Evaluate on all sets**


In [ ]:
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

val_mae = mean_absolute_error(y_val, y_val_pred)
val_mse = mean_squared_error(y_val, y_val_pred)
val_r2 = r2_score(y_val, y_val_pred)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("=" * 25)
print("Train Set:")
print(f"MAE  : {train_mae:.4f}")
print(f"MSE  : {train_mse:.4f}")
print(f"R²   : {train_r2:.4f}")
print("\nValidation Set:")
print(f"MAE  : {val_mae:.4f}")
print(f"MSE  : {val_mse:.4f}")
print(f"R²   : {val_r2:.4f}")
print("\nTest Set:")
print(f"MAE  : {test_mae:.4f}")
print(f"MSE  : {test_mse:.4f}")
print(f"R²   : {test_r2:.4f}")
print("=" * 25)

Train Set:
MAE  : 2.5256
MSE  : 11.1093
R²   : 0.9084

Validation Set:
MAE  : 6.8179
MSE  : 80.2848
R²   : 0.3404

Test Set:
MAE  : 6.8285
MSE  : 80.2516
R²   : 0.3477


In [ ]:
# Store results
model_results['RF_Initial'] = {
    'Train_R2': train_r2, 'Train_MAE': train_mae, 'Train_MSE': train_mse,
    'Val_R2': val_r2, 'Val_MAE': val_mae, 'Val_MSE': val_mse,
    'Test_R2': test_r2, 'Test_MAE': test_mae, 'Test_MSE': test_mse
}

### **Analyze overfitting**


In [ ]:
print("\nOverfitting Analysis:")
if val_mse > train_mse * 1.2 or test_mse > train_mse * 1.2:
    print("Warning: Potential overfitting detected! Validation or Test MSE is significantly higher than Train MSE.")
else:
    print("No significant overfitting detected. Train, Validation, and Test MSE are relatively close.")



Overfitting Analysis:


## **Grid Search for hyperparameter tuning**


In [ ]:
print("\nGrid Search for Hyperparameter Tuning:")
print("=" * 42)

param_grid = {
    'n_estimators': [25, 50, 75],
    'max_depth': [5, 10],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 6]
}

# Perform Grid Search
grid_search = GridSearchCV(estimator=RandomForestRegressor(random_state=42),
                           param_grid=param_grid,
                           cv=5,
                           scoring='neg_mean_squared_error',
                           n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best parameters and score
best_params = grid_search.best_params_
best_val_mse = -grid_search.best_score_

print("\nBest Hyperparameters:")
print(f"Best Parameters: {best_params}")
print(f"Best Validation MSE: {best_val_mse:.4f}\n")
print("=" * 42)

# Grid Search for Hyperparameter Tuning:
# ==========================================

# Best Hyperparameters:
# Best Parameters: {'max_depth': 20, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 100}
# Best Validation MSE: 78.2200

# ==========================================


Grid Search for Hyperparameter Tuning:

Best Hyperparameters:
Best Parameters: {'max_depth': 20, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 100}
Best Validation MSE: 78.2200



## **Train final model with best hyperparameters**


In [12]:
print("\nTraining Final Model with Best Hyperparameters:")
print("=" * 55)

final_model = RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_split=10, min_samples_leaf=4, random_state=42)
final_model.fit(X_train, y_train)


Training Final Model with Best Hyperparameters:


RandomForestRegressor(max_depth=20, min_samples_leaf=4, min_samples_split=10,
                      random_state=42)

### **Predict with final model on test set**


In [13]:
y_test_pred_final = final_model.predict(X_test)

### **Final evaluation on test set**
test_mae_final = mean_absolute_error(y_test, y_test_pred_final)
test_mse_final = mean_squared_error(y_test, y_test_pred_final)
test_r2_final = r2_score(y_test, y_test_pred_final)
test_rmse_final = np.sqrt(test_mse_final)

# Calculate additional metrics for final model
test_mpe_final = calculate_mpe(y_test, y_test_pred_final)
test_smape_final = calculate_smape(y_test, y_test_pred_final)
test_sdr_final = calculate_sdr(y_test, y_test_pred_final)

print("Final Model Evaluation (Test Set - RF_Tuned):")
print("=" * 60)
print(f"R²    : {test_r2_final:.4f}")
print(f"MAE   : {test_mae_final:.4f}")
print(f"RMSE  : {test_rmse_final:.4f}")
print(f"MPE   : {test_mpe_final:.4f}")
print(f"sMAPE : {test_smape_final:.4f}")
print(f"SDR   : {test_sdr_final:.4f}")
print("=" * 60)

Final Model Evaluation (Test Set - RF_Tuned):
R²    : 0.3575
MAE   : 6.7591
RMSE  : 8.8906
MPE   : -7.3158
sMAPE : 17.0551
SDR   : 0.6032


In [19]:
## **Cross-Validation for Random Forest**
def cross_validate_rf_with_overfitting_check(X, y, X_test, y_test, k=5, **params):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    train_r2_scores = []
    val_r2_scores = []
    train_mae_scores = []
    val_mae_scores = []
    train_mse_scores = []
    val_mse_scores = []

    fold = 1
    for train_index, val_index in kf.split(X):
        X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
        y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

        # Pass params without duplicating random_state
        model_params = params.copy()
        model_params['random_state'] = 42  # Ensure random_state is set once
        model = RandomForestRegressor(**model_params)
        model.fit(X_train_fold, y_train_fold)

        y_train_pred = model.predict(X_train_fold)
        y_val_pred = model.predict(X_val_fold)

        train_r2 = r2_score(y_train_fold, y_train_pred)
        val_r2 = r2_score(y_val_fold, y_val_pred)
        train_mae = mean_absolute_error(y_train_fold, y_train_pred)
        val_mae = mean_absolute_error(y_val_fold, y_val_pred)
        train_mse = mean_squared_error(y_train_fold, y_train_pred)
        val_mse = mean_squared_error(y_val_fold, y_val_pred)

        train_r2_scores.append(train_r2)
        val_r2_scores.append(val_r2)
        train_mae_scores.append(train_mae)
        val_mae_scores.append(val_mae)
        train_mse_scores.append(train_mse)
        val_mse_scores.append(val_mse)

        fold += 1

    # Train final model on full training data and evaluate on Test Set
    final_model = RandomForestRegressor(**params, random_state=42)
    final_model.fit(X, y)
    y_test_pred = final_model.predict(X_test)
    test_r2 = r2_score(y_test, y_test_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    test_rmse = np.sqrt(test_mse)
    test_mpe = calculate_mpe(y_test, y_test_pred)
    test_smape = calculate_smape(y_test, y_test_pred)
    test_sdr = calculate_sdr(y_test, y_test_pred)

    return final_model, test_r2, test_mae, test_mse, test_rmse, test_mpe, test_smape, test_sdr

# Run Cross-Validation and get the final model results
final_model_cv, test_r2_cv, test_mae_cv, test_mse_cv, test_rmse_cv, test_mpe_cv, test_smape_cv, test_sdr_cv = cross_validate_rf_with_overfitting_check(
    X_train, y_train, X_test, y_test,
    max_depth=10, min_samples_leaf=6, min_samples_split=15, n_estimators=75
)

print("Cross-Validation Final Model Evaluation (Test Set - RF_CV_Final):")
print("=" * 60)
print(f"R²    : {test_r2_cv:.4f}")
print(f"MAE   : {test_mae_cv:.4f}")
print(f"RMSE  : {test_rmse_cv:.4f}")
print(f"MPE   : {test_mpe_cv:.4f}")
print(f"sMAPE : {test_smape_cv:.4f}")
print(f"SDR   : {test_sdr_cv:.4f}")
print("=" * 60)

Cross-Validation Final Model Evaluation (Test Set - RF_CV_Final):
R²    : 0.3526
MAE   : 6.7850
RMSE  : 8.9250
MPE   : -7.6106
sMAPE : 17.0927
SDR   : 0.5744


### **Predict with final model**


In [ ]:
y_train_pred_final = final_model.predict(X_train)
y_val_pred_final = final_model.predict(X_val)
y_test_pred_final = final_model.predict(X_test)

### **Final evaluation**


In [ ]:
train_mae_final = mean_absolute_error(y_train, y_train_pred_final)
train_mse_final = mean_squared_error(y_train, y_train_pred_final)
train_r2_final = r2_score(y_train, y_train_pred_final)

val_mae_final = mean_absolute_error(y_val, y_val_pred_final)
val_mse_final = mean_squared_error(y_val, y_val_pred_final)
val_r2_final = r2_score(y_val, y_val_pred_final)

test_mae_final = mean_absolute_error(y_test, y_test_pred_final)
test_mse_final = mean_squared_error(y_test, y_test_pred_final)
test_r2_final = r2_score(y_test, y_test_pred_final)

# Display final results
print("Final Model Evaluation:")
print("=" * 60)
print("Train Set:")
print(f"MAE  : {train_mae_final:.4f}")
print(f"MSE  : {train_mse_final:.4f}")
print(f"R²   : {train_r2_final:.4f}")
print("\nValidation Set:")
print(f"MAE  : {val_mae_final:.4f}")
print(f"MSE  : {val_mse_final:.4f}")
print(f"R²   : {val_r2_final:.4f}")
print("\nTest Set:")
print(f"MAE  : {test_mae_final:.4f}")
print(f"MSE  : {test_mse_final:.4f}")
print(f"R²   : {test_r2_final:.4f}")
print("=" * 60)

Final Model Evaluation:
Train Set:
MAE  : 6.6120
MSE  : 75.2452
R²   : 0.3796

Validation Set:
MAE  : 6.7722
MSE  : 79.6481
R²   : 0.3457

Test Set:
MAE  : 6.7850
MSE  : 79.6555
R²   : 0.3526


In [ ]:
# Store results
model_results['RF_Tuned'] = {
    'Train_R2': train_r2_final, 'Train_MAE': train_mae_final, 'Train_MSE': train_mse_final,
    'Val_R2': val_r2_final, 'Val_MAE': val_mae_final, 'Val_MSE': val_mse_final,
    'Test_R2': test_r2_final, 'Test_MAE': test_mae_final, 'Test_MSE': test_mse_final
}

## **Cross-Validation for Random Forest**

In [ ]:
def cross_validate_rf_with_overfitting_check(X, y, X_test, y_test, k=5, max_depth=10, min_samples_leaf=6, min_samples_split=15, n_estimators=75, random_state=42):
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    train_r2_scores = []
    val_r2_scores = []
    train_mae_scores = []
    val_mae_scores = []
    train_mse_scores = []
    val_mse_scores = []

    fold = 1
    for train_index, val_index in kf.split(X):
        X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
        y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            min_samples_split=min_samples_split,
            random_state=random_state
        )
        model.fit(X_train_fold, y_train_fold)

        y_train_pred = model.predict(X_train_fold)
        y_val_pred = model.predict(X_val_fold)

        train_r2 = r2_score(y_train_fold, y_train_pred)
        val_r2 = r2_score(y_val_fold, y_val_pred)
        train_mae = mean_absolute_error(y_train_fold, y_train_pred)
        val_mae = mean_absolute_error(y_val_fold, y_val_pred)
        train_mse = mean_squared_error(y_train_fold, y_train_pred)
        val_mse = mean_squared_error(y_val_fold, y_val_pred)

        train_r2_scores.append(train_r2)
        val_r2_scores.append(val_r2)
        train_mae_scores.append(train_mae)
        val_mae_scores.append(val_mae)
        train_mse_scores.append(train_mse)
        val_mse_scores.append(val_mse)

        print(f"Fold {fold}: Train R² = {train_r2:.4f}, Val R² = {val_r2:.4f}")
        print(f"Fold {fold}: Train MAE = {train_mae:.4f}, Val MAE = {val_mae:.4f}")
        print(f"Fold {fold}: Train MSE = {train_mse:.4f}, Val MSE = {val_mse:.4f}")
        print(f"Fold {fold}: Train R² Std: {np.std(train_r2_scores):.4f}")
        print(f"Fold {fold}: Val R² Std: {np.std(val_r2_scores):.4f}")
        print("=" * 60)

        fold += 1

    avg_train_r2 = np.mean(train_r2_scores)
    avg_val_r2 = np.mean(val_r2_scores)
    r2_gap = avg_train_r2 - avg_val_r2
    train_r2_std = np.std(train_r2_scores)
    val_r2_std = np.std(val_r2_scores)
    avg_train_mae = np.mean(train_mae_scores)
    avg_val_mae = np.mean(val_mae_scores)
    avg_train_mse = np.mean(train_mse_scores)
    avg_val_mse = np.mean(val_mse_scores)

    print(f"\nAverage Train R²: {avg_train_r2:.4f}")
    print(f"Average Val R²: {avg_val_r2:.4f}")
    print(f"Train - Val R² Gap: {r2_gap:.4f}")
    print(f"Train R² Std: {train_r2_std:.4f}")
    print(f"Val R² Std: {val_r2_std:.4f}")
    print(f"Average Train MAE: {avg_train_mae:.4f}")
    print(f"Average Val MAE: {avg_val_mae:.4f}")
    print(f"Average Train MSE: {avg_train_mse:.4f}")
    print(f"Average Val MSE: {avg_val_mse:.4f}")

    print("\nOverfitting Analysis:")
    r2_gap_threshold = 0.05
    val_r2_std_threshold = 0.02
    if r2_gap > r2_gap_threshold:
        print(f"Warning: Potential Overfitting Detected! Train-Val R² Gap ({r2_gap:.4f}) is larger than threshold ({r2_gap_threshold}).")
    if val_r2_std > val_r2_std_threshold:
        print(f"Warning: Model performance is unstable! Val R² Std ({val_r2_std:.4f}) is larger than threshold ({val_r2_std_threshold}).")

    # Train final model on full training data (X, y) and evaluate on Test Set
    final_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        random_state=random_state
    )
    final_model.fit(X, y)
    y_test_pred = final_model.predict(X_test)
    test_r2 = r2_score(y_test, y_test_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    print(f"\nTest Set Evaluation (Final Model after CV):")
    print(f"Test R²: {test_r2:.4f}")
    print(f"Test MAE: {test_mae:.4f}")
    print(f"Test MSE: {test_mse:.4f}")

    test_r2_drop_threshold = 0.05
    if avg_val_r2 - test_r2 > test_r2_drop_threshold:
        print(f"Warning: Potential Overfitting Detected! Test R² ({test_r2:.4f}) dropped significantly compared to Avg Val R² ({avg_val_r2:.4f}).")
    else:
        print(f"No significant overfitting based on Test Set performance. Test R² ({test_r2:.4f}) is consistent with Avg Val R² ({avg_val_r2:.4f}).")

    return final_model, train_r2_scores, val_r2_scores, train_mae_scores, val_mae_scores, train_mse_scores, val_mse_scores, cv_final_results

In [ ]:
# Run Cross-Validation and get the final model results
final_model, train_r2_scores, val_r2_scores, train_mae_scores, val_mae_scores, train_mse_scores, val_mse_scores, cv_final_results = cross_validate_rf_with_overfitting_check(X_train, y_train, X_test, y_test)


Fold 1: Train R² = 0.3836, Val R² = 0.3518
Fold 1: Train MAE = 6.5844, Val MAE = 6.7760
Fold 1: Train MSE = 74.6968, Val MSE = 78.8709
Fold 1: Train R² Std: 0.0000
Fold 1: Val R² Std: 0.0000
Fold 2: Train R² = 0.3839, Val R² = 0.3531
Fold 2: Train MAE = 6.5898, Val MAE = 6.7501
Fold 2: Train MSE = 74.7324, Val MSE = 78.3885
Fold 2: Train R² Std: 0.0002
Fold 2: Val R² Std: 0.0006
Fold 3: Train R² = 0.3850, Val R² = 0.3515
Fold 3: Train MAE = 6.5851, Val MAE = 6.7404
Fold 3: Train MSE = 74.6201, Val MSE = 78.5190
Fold 3: Train R² Std: 0.0006
Fold 3: Val R² Std: 0.0007
Fold 4: Train R² = 0.3860, Val R² = 0.3469
Fold 4: Train MAE = 6.5737, Val MAE = 6.7876
Fold 4: Train MSE = 74.2795, Val MSE = 79.9806
Fold 4: Train R² Std: 0.0010
Fold 4: Val R² Std: 0.0024
Fold 5: Train R² = 0.3838, Val R² = 0.3506
Fold 5: Train MAE = 6.5974, Val MAE = 6.7339
Fold 5: Train MSE = 74.9271, Val MSE = 77.9156
Fold 5: Train R² Std: 0.0009
Fold 5: Val R² Std: 0.0021

Average Train R²: 0.3845
Average Val R²: 0.3

# **Random forest Use grp**

In [20]:
df_grp = pd.read_excel('E:\IUT\Lessons\Project-Bachelor\Husbandry\Dataset\TCI_sas (1).xlsx', sheet_name="Sheet1", usecols = ["firstmilk","firstmilkdays",
                                                                "prelendays","drylendays","milkdays",
                                                                "grp", "previous_Milk305",
                                                                "firstmilk_previous","SCS_305"])

In [21]:
df_grp['drylendays'] = df_grp['drylendays'].fillna(df_grp['drylendays'].mean())
df_grp['milkdays'] = df_grp['milkdays'].fillna(df_grp['milkdays'].mean())

In [22]:
# Separate features and target
X_grp = df_grp.drop(columns=["firstmilk"])
y_grp = df_grp["firstmilk"].astype(float) 

# First split: separate test set (15%)
X_temp_grp, X_test_grp, y_temp_grp, y_test_grp = train_test_split(X_grp, y_grp, test_size=0.15, random_state=42)

# Second split: separate train (70%) and validation (15%) from remaining data
X_train_grp, X_val_grp, y_train_grp, y_val_grp = train_test_split(X_temp_grp, y_temp_grp, test_size=0.1765, random_state=42)  # 0.1765 ~ 15/(100-15)


In [23]:
numerical_columns = ["firstmilkdays", "prelendays", "drylendays", "milkdays","grp",
                     "previous_Milk305", "firstmilk_previous", "SCS_305"]

scaler = StandardScaler()

# Fit scaler on training data only
scaler.fit(X_train_grp[numerical_columns])

# Transform train, validation, and test sets
X_train_grp[numerical_columns] = scaler.transform(X_train_grp[numerical_columns])
X_val_grp[numerical_columns] = scaler.transform(X_val_grp[numerical_columns])
X_test_grp[numerical_columns] = scaler.transform(X_test_grp[numerical_columns])

# **Random Forest Initial with 'grp' (Default Parameters)**

In [24]:
print("\n=== Random Forest Initial with 'grp' (Default Parameters) ===")
initial_model_grp = RandomForestRegressor(random_state=42)
initial_model_grp.fit(X_train_grp, y_train_grp)


y_test_pred_initial_grp = initial_model_grp.predict(X_test_grp)

test_mae_initial_grp = mean_absolute_error(y_test_grp, y_test_pred_initial_grp)
test_mse_initial_grp = mean_squared_error(y_test_grp, y_test_pred_initial_grp)
test_r2_initial_grp = r2_score(y_test_grp, y_test_pred_initial_grp)
test_rmse_initial_grp = np.sqrt(test_mse_initial_grp)
test_mpe_initial_grp = calculate_mpe(y_test_grp, y_test_pred_initial_grp)
test_smape_initial_grp = calculate_smape(y_test_grp, y_test_pred_initial_grp)
test_sdr_initial_grp = calculate_sdr(y_test_grp, y_test_pred_initial_grp)

print("Test Set (Initial with 'grp'):")
print(f"R²    : {test_r2_initial_grp:.4f}")
print(f"MAE   : {test_mae_initial_grp:.4f}")
print(f"RMSE  : {test_rmse_initial_grp:.4f}")
print(f"MPE   : {test_mpe_initial_grp:.4f}")
print(f"sMAPE : {test_smape_initial_grp:.4f}")
print(f"SDR   : {test_sdr_initial_grp:.4f}")


=== Random Forest Initial with 'grp' (Default Parameters) ===
Test Set (Initial with 'grp'):
R²    : 0.3401
MAE   : 6.8785
RMSE  : 9.0107
MPE   : -6.9643
sMAPE : 17.3424
SDR   : 0.6136


In [ ]:
print("\n=== Random Forest Initial with 'grp' (Default Parameters) ===")
initial_model_grp = RandomForestRegressor(random_state=42)
initial_model_grp.fit(X_train_grp, y_train_grp)

y_train_pred_initial_grp = initial_model_grp.predict(X_train_grp)
y_val_pred_initial_grp = initial_model_grp.predict(X_val_grp)
y_test_pred_initial_grp = initial_model_grp.predict(X_test_grp)

train_mae_initial_grp = mean_absolute_error(y_train_grp, y_train_pred_initial_grp)
train_mse_initial_grp = mean_squared_error(y_train_grp, y_train_pred_initial_grp)
train_r2_initial_grp = r2_score(y_train_grp, y_train_pred_initial_grp)
val_mae_initial_grp = mean_absolute_error(y_val_grp, y_val_pred_initial_grp)
val_mse_initial_grp = mean_squared_error(y_val_grp, y_val_pred_initial_grp)
val_r2_initial_grp = r2_score(y_val_grp, y_val_pred_initial_grp)
test_mae_initial_grp = mean_absolute_error(y_test_grp, y_test_pred_initial_grp)
test_mse_initial_grp = mean_squared_error(y_test_grp, y_test_pred_initial_grp)
test_r2_initial_grp = r2_score(y_test_grp, y_test_pred_initial_grp)

print("Train Set (Initial with 'grp'):")
print(f"MAE  : {train_mae_initial_grp:.4f}")
print(f"MSE  : {train_mse_initial_grp:.4f}")
print(f"R²   : {train_r2_initial_grp:.4f}")
print("\nValidation Set (Initial with 'grp'):")
print(f"MAE  : {val_mae_initial_grp:.4f}")
print(f"MSE  : {val_mse_initial_grp:.4f}")
print(f"R²   : {val_r2_initial_grp:.4f}")
print("\nTest Set (Initial with 'grp'):")
print(f"MAE  : {test_mae_initial_grp:.4f}")
print(f"MSE  : {test_mse_initial_grp:.4f}")
print(f"R²   : {test_r2_initial_grp:.4f}")

# Store results
model_results['RF_Initial_with_grp'] = {
    'Train_R2': train_r2_initial_grp, 'Train_MAE': train_mae_initial_grp, 'Train_MSE': train_mse_initial_grp,
    'Val_R2': val_r2_initial_grp, 'Val_MAE': val_mae_initial_grp, 'Val_MSE': val_mse_initial_grp,
    'Test_R2': test_r2_initial_grp, 'Test_MAE': test_mae_initial_grp, 'Test_MSE': test_mse_initial_grp
}

# Overfitting Analysis for Initial Model with 'grp'
print("\nOverfitting Analysis (Initial with 'grp'):")
if val_mse_initial_grp > train_mse_initial_grp * 1.2 or test_mse_initial_grp > train_mse_initial_grp * 1.2:
    print("Warning: Potential overfitting detected! Validation or Test MSE is significantly higher than Train MSE.")
else:
    print("No significant overfitting detected. Train, Validation, and Test MSE are relatively close.") 


=== Random Forest Initial with 'grp' (Default Parameters) ===
Train Set (Initial with 'grp'):
MAE  : 2.5412
MSE  : 11.2257
R²   : 0.9074

Validation Set (Initial with 'grp'):
MAE  : 6.8535
MSE  : 81.0523
R²   : 0.3341

Test Set (Initial with 'grp'):
MAE  : 6.8785
MSE  : 81.1934
R²   : 0.3401

Overfitting Analysis (Initial with 'grp'):


## **Grid Search for Random Forest with 'grp'**

In [ ]:
print("\nGrid Search for Hyperparameter Tuning (RF with 'grp'):")
param_grid_grp = {
    'n_estimators': [25, 50, 75],
    'max_depth': [5, 10],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 6]
}

grid_search_grp = GridSearchCV(estimator=RandomForestRegressor(random_state=42),
                               param_grid=param_grid_grp,
                               cv=5,
                               scoring='neg_mean_squared_error',
                               n_jobs=-1)
grid_search_grp.fit(X_train_grp, y_train_grp)

best_params_grp = grid_search_grp.best_params_
best_val_mse_grp = -grid_search_grp.best_score_
print("\nBest Hyperparameters (RF with 'grp'):")
print(f"Best Parameters: {best_params_grp}")
print(f"Best Validation MSE: {best_val_mse_grp:.4f}\n")

# Grid Search for Hyperparameter Tuning (RF with 'grp'):

# Best Hyperparameters (RF with 'grp'):
# Best Parameters: {'max_depth': 10, 'min_samples_leaf': 6, 'min_samples_split': 15, 'n_estimators': 75}
# Best Validation MSE: 78.8758

## **Train final model with best hyperparameters (with 'grp')**

In [25]:
# Assuming best parameters from your comment
best_params_grp = {'max_depth': 10, 'min_samples_leaf': 6, 'min_samples_split': 15, 'n_estimators': 75}

## **Train final model with best hyperparameters (with 'grp')**
print("\nTraining Final Model with Best Hyperparameters (with 'grp'):")
final_model_grp = RandomForestRegressor(max_depth=best_params_grp['max_depth'],
                                       min_samples_split=best_params_grp['min_samples_split'],
                                       min_samples_leaf=best_params_grp['min_samples_leaf'],
                                       n_estimators=best_params_grp['n_estimators'],
                                       random_state=42)
final_model_grp.fit(X_train_grp, y_train_grp)

y_test_pred_final_grp = final_model_grp.predict(X_test_grp)

test_mae_final_grp = mean_absolute_error(y_test_grp, y_test_pred_final_grp)
test_mse_final_grp = mean_squared_error(y_test_grp, y_test_pred_final_grp)
test_r2_final_grp = r2_score(y_test_grp, y_test_pred_final_grp)
test_rmse_final_grp = np.sqrt(test_mse_final_grp)
test_mpe_final_grp = calculate_mpe(y_test_grp, y_test_pred_final_grp)
test_smape_final_grp = calculate_smape(y_test_grp, y_test_pred_final_grp)
test_sdr_final_grp = calculate_sdr(y_test_grp, y_test_pred_final_grp)

print("Final Model Evaluation (Test Set - RF_with_grp):")
print("Test Set:")
print(f"R²    : {test_r2_final_grp:.4f}")
print(f"MAE   : {test_mae_final_grp:.4f}")
print(f"RMSE  : {test_rmse_final_grp:.4f}")
print(f"MPE   : {test_mpe_final_grp:.4f}")
print(f"sMAPE : {test_smape_final_grp:.4f}")
print(f"SDR   : {test_sdr_final_grp:.4f}")


Training Final Model with Best Hyperparameters (with 'grp'):
Final Model Evaluation (Test Set - RF_with_grp):
Test Set:
R²    : 0.3514
MAE   : 6.7925
RMSE  : 8.9331
MPE   : -7.6089
sMAPE : 17.1123
SDR   : 0.5749


In [ ]:
print("\nTraining Final Model with Best Hyperparameters (with 'grp'):")
# final_model_grp = RandomForestRegressor(**best_params_grp, random_state=42)
final_model_grp = RandomForestRegressor(max_depth=10, min_samples_split=15, min_samples_leaf=6, n_estimators=75, random_state=42)

final_model_grp.fit(X_train_grp, y_train_grp)

y_train_pred_final_grp = final_model_grp.predict(X_train_grp)
y_val_pred_final_grp = final_model_grp.predict(X_val_grp)
y_test_pred_final_grp = final_model_grp.predict(X_test_grp)

train_mae_final_grp = mean_absolute_error(y_train_grp, y_train_pred_final_grp)
train_mse_final_grp = mean_squared_error(y_train_grp, y_train_pred_final_grp)
train_r2_final_grp = r2_score(y_train_grp, y_train_pred_final_grp)
val_mae_final_grp = mean_absolute_error(y_val_grp, y_val_pred_final_grp)
val_mse_final_grp = mean_squared_error(y_val_grp, y_val_pred_final_grp)
val_r2_final_grp = r2_score(y_val_grp, y_val_pred_final_grp)
test_mae_final_grp = mean_absolute_error(y_test_grp, y_test_pred_final_grp)
test_mse_final_grp = mean_squared_error(y_test_grp, y_test_pred_final_grp)
test_r2_final_grp = r2_score(y_test_grp, y_test_pred_final_grp)

print("Final Model Evaluation (with 'grp'):")
print("Train Set:")
print(f"MAE  : {train_mae_final_grp:.4f}")
print(f"MSE  : {train_mse_final_grp:.4f}")
print(f"R²   : {train_r2_final_grp:.4f}")
print("\nValidation Set:")
print(f"MAE  : {val_mae_final_grp:.4f}")
print(f"MSE  : {val_mse_final_grp:.4f}")
print(f"R²   : {val_r2_final_grp:.4f}")
print("\nTest Set:")
print(f"MAE  : {test_mae_final_grp:.4f}")
print(f"MSE  : {test_mse_final_grp:.4f}")
print(f"R²   : {test_r2_final_grp:.4f}")

# Store results
model_results['RF_with_grp'] = {
    'Train_R2': train_r2_final_grp, 'Train_MAE': train_mae_final_grp, 'Train_MSE': train_mse_final_grp,
    'Val_R2': val_r2_final_grp, 'Val_MAE': val_mae_final_grp, 'Val_MSE': val_mse_final_grp,
    'Test_R2': test_r2_final_grp, 'Test_MAE': test_mae_final_grp, 'Test_MSE': test_mse_final_grp
}


Training Final Model with Best Hyperparameters (with 'grp'):
Final Model Evaluation (with 'grp'):
Train Set:
MAE  : 6.6208
MSE  : 75.4154
R²   : 0.3782

Validation Set:
MAE  : 6.7796
MSE  : 79.7673
R²   : 0.3447

Test Set:
MAE  : 6.7925
MSE  : 79.8008
R²   : 0.3514


## **Cross-Validation for Random Forest with 'grp'**


In [26]:
## **Cross-Validation for Random Forest with 'grp'**
print("\nCross-Validation for Random Forest with 'grp':")
final_model_grp_cv, test_r2_cv_grp, test_mae_cv_grp, test_mse_cv_grp, test_rmse_cv_grp, test_mpe_cv_grp, test_smape_cv_grp, test_sdr_cv_grp = cross_validate_rf_with_overfitting_check(
    X_train_grp, y_train_grp, X_test_grp, y_test_grp,
    max_depth=10, min_samples_leaf=6, min_samples_split=15, n_estimators=75
)

print("Cross-Validation Final Model Evaluation (Test Set - RF_with_grp_CV_Final):")
print("=" * 60)
print(f"R²    : {test_r2_cv_grp:.4f}")
print(f"MAE   : {test_mae_cv_grp:.4f}")
print(f"RMSE  : {test_rmse_cv_grp:.4f}")
print(f"MPE   : {test_mpe_cv_grp:.4f}")
print(f"sMAPE : {test_smape_cv_grp:.4f}")
print(f"SDR   : {test_sdr_cv_grp:.4f}")
print("=" * 60)


Cross-Validation for Random Forest with 'grp':
Cross-Validation Final Model Evaluation (Test Set - RF_with_grp_CV_Final):
R²    : 0.3514
MAE   : 6.7925
RMSE  : 8.9331
MPE   : -7.6089
sMAPE : 17.1123
SDR   : 0.5749


In [ ]:
print("\nCross-Validation for Random Forest with 'grp':")
# final_model_grp_cv, train_r2_scores_grp, val_r2_scores_grp, train_mae_scores_grp, val_mae_scores_grp, train_mse_scores_grp, val_mse_scores_grp, cv_final_results_grp = cross_validate_rf_with_overfitting_check(
#     X_train_grp, y_train_grp, X_test_grp, y_test_grp,
#     max_depth=best_params_grp['max_depth'],
#     min_samples_leaf=best_params_grp['min_samples_leaf'],
#     min_samples_split=best_params_grp['min_samples_split'],
#     n_estimators=best_params_grp['n_estimators'],
#     random_state=42
# )
final_model_grp_cv, train_r2_scores_grp, val_r2_scores_grp, train_mae_scores_grp, val_mae_scores_grp, train_mse_scores_grp, val_mse_scores_grp, cv_final_results_grp = cross_validate_rf_with_overfitting_check(
    X_train_grp, y_train_grp, X_test_grp, y_test_grp,
    max_depth=10,
    min_samples_leaf=6,
    min_samples_split=15,
    n_estimators=75,
    random_state=42
)


Cross-Validation for Random Forest with 'grp':
Fold 1: Train R² = 0.3817, Val R² = 0.3507
Fold 1: Train MAE = 6.5956, Val MAE = 6.7820
Fold 1: Train MSE = 74.9206, Val MSE = 79.0143
Fold 1: Train R² Std: 0.0000
Fold 1: Val R² Std: 0.0000
Fold 2: Train R² = 0.3823, Val R² = 0.3518
Fold 2: Train MAE = 6.5998, Val MAE = 6.7574
Fold 2: Train MSE = 74.9277, Val MSE = 78.5423
Fold 2: Train R² Std: 0.0003
Fold 2: Val R² Std: 0.0006
Fold 3: Train R² = 0.3834, Val R² = 0.3505
Fold 3: Train MAE = 6.5947, Val MAE = 6.7472
Fold 3: Train MSE = 74.8047, Val MSE = 78.6509
Fold 3: Train R² Std: 0.0007
Fold 3: Val R² Std: 0.0006
Fold 4: Train R² = 0.3844, Val R² = 0.3458
Fold 4: Train MAE = 6.5835, Val MAE = 6.7935
Fold 4: Train MSE = 74.4761, Val MSE = 80.1138
Fold 4: Train R² Std: 0.0010
Fold 4: Val R² Std: 0.0023
Fold 5: Train R² = 0.3822, Val R² = 0.3500
Fold 5: Train MAE = 6.6077, Val MAE = 6.7377
Fold 5: Train MSE = 75.1295, Val MSE = 77.9801
Fold 5: Train R² Std: 0.0010
Fold 5: Val R² Std: 0.00

In [ ]:
# Add the CV final model (with 'grp') to the results
model_results['RF_with_grp_CV_Final'] = cv_final_results_grp

In [27]:
## **Print results into a new CSV file**
# Function to print existing CSV content
def print_csv_content(file_path):
    try:
        existing_df = pd.read_csv(file_path, index_col=0)
        return existing_df
    except FileNotFoundError:
        print("\nNo existing test evaluation table found.")
        return pd.DataFrame()

# Load existing CSV content from new file (if exists)
new_file_path = 'test_evaluation_metrics.csv'
existing_df = print_csv_content(new_file_path)

# Save results to a new CSV file
initial_results = {
    'RF_Initial': {
        'Test_R2': test_r2, 'Test_MAE': test_mae, 'Test_RMSE': test_rmse,
        'Test_MPE': test_mpe, 'Test_sMAPE': test_smape, 'Test_SDR': test_sdr
    }
}
initial_df = pd.DataFrame.from_dict(initial_results, orient='index')

final_results = {
    'RF_Tuned': {
        'Test_R2': test_r2_final, 'Test_MAE': test_mae_final, 'Test_RMSE': test_rmse_final,
        'Test_MPE': test_mpe_final, 'Test_sMAPE': test_smape_final, 'Test_SDR': test_sdr_final
    }
}
final_df = pd.DataFrame.from_dict(final_results, orient='index')

cv_results = {
    'RF_CV_Final': {
        'Test_R2': test_r2_cv, 'Test_MAE': test_mae_cv, 'Test_RMSE': test_rmse_cv,
        'Test_MPE': test_mpe_cv, 'Test_sMAPE': test_smape_cv, 'Test_SDR': test_sdr_cv
    }
}
cv_df = pd.DataFrame.from_dict(cv_results, orient='index')

initial_results_grp = {
    'RF_Initial_with_grp': {
        'Test_R2': test_r2_initial_grp, 'Test_MAE': test_mae_initial_grp, 'Test_RMSE': test_rmse_initial_grp,
        'Test_MPE': test_mpe_initial_grp, 'Test_sMAPE': test_smape_initial_grp, 'Test_SDR': test_sdr_initial_grp
    }
}
initial_df_grp = pd.DataFrame.from_dict(initial_results_grp, orient='index')

final_results_grp = {
    'RF_with_grp': {
        'Test_R2': test_r2_final_grp, 'Test_MAE': test_mae_final_grp, 'Test_RMSE': test_rmse_final_grp,
        'Test_MPE': test_mpe_final_grp, 'Test_sMAPE': test_smape_final_grp, 'Test_SDR': test_sdr_final_grp
    }
}
final_df_grp = pd.DataFrame.from_dict(final_results_grp, orient='index')

cv_results_grp = {
    'RF_with_grp_CV_Final': {
        'Test_R2': test_r2_cv_grp, 'Test_MAE': test_mae_cv_grp, 'Test_RMSE': test_rmse_cv_grp,
        'Test_MPE': test_mpe_cv_grp, 'Test_sMAPE': test_smape_cv_grp, 'Test_SDR': test_sdr_cv_grp
    }
}
cv_df_grp = pd.DataFrame.from_dict(cv_results_grp, orient='index')

# Combine new results
new_results_df = pd.concat([initial_df, final_df, cv_df, initial_df_grp, final_df_grp, cv_df_grp])

# If existing data exists, append it; otherwise, start with new results
if not existing_df.empty:
    comparison_df = pd.concat([new_results_df, existing_df])
else:
    comparison_df = new_results_df

# Save to the new file
comparison_df.to_csv(new_file_path, index=True)
print("\n=== Updated Test Evaluation Metrics Saved to test_evaluation_metrics.csv ===")
print(comparison_df)


=== Updated Test Evaluation Metrics Saved to test_evaluation_metrics.csv ===
                       Test_R2  Test_MAE  Test_RMSE  Test_MPE  Test_sMAPE  \
RF_Initial            0.347709  6.828523   8.958325 -6.936344   17.222969   
RF_Tuned              0.357528  6.759105   8.890640 -7.315823   17.055061   
RF_CV_Final           0.352554  6.784989   8.924994 -7.610628   17.092712   
RF_Initial_with_grp   0.340054  6.878539   9.010737 -6.964281   17.342442   
RF_with_grp           0.351373  6.792523   8.933128 -7.608877   17.112260   
RF_with_grp_CV_Final  0.351373  6.792523   8.933128 -7.608877   17.112260   
Ridge_Initial         0.310995  7.027556   9.206984 -8.085323   17.683976   
Ridge_Optimized       0.313395  7.005673   9.190935 -7.877927   17.664176   

                      Test_SDR  
RF_Initial            0.610604  
RF_Tuned              0.603169  
RF_CV_Final           0.574371  
RF_Initial_with_grp   0.613634  
RF_with_grp           0.574870  
RF_with_grp_CV_Final  0.574870